# Player Time on Ice Exploration

In [67]:
# Dependencies

# Basics
import os
import sys
import time
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from collections import defaultdict
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path


from config import recent_clean_db, last_game_date


In [68]:
## Test color dictionary

### Team info Dictionaries
team_color_mapping = {'Air Force': ('#003087', '#8a8d8f', '#CCCCCC'),
 'Alaska': ('#236192', '#ffcd00', '#CCCCCC'),
 'Alaska Anchorage': ('#00583d', '#ffc425', '#CCCCCC'),
 'American Intl': ('#000000', '#ffb60f', '#CCCCCC'),
 "American Int'l": ('#000000', '#ffb60f', '#CCCCCC'),
 'Arizona State': ('#8c1d40', '#ffc627', '#CCCCCC'),
 'Army': ('#d4bf91', '#b2b4b3', '#CCCCCC'),
 'Augustana': ('#004b8d', '#ffdd00', '#FFFFFF'),
 'Bemidji State': ('#004d44', '#d4d67c', '#CCCCCC'),
 'Bentley': ('#1b5faa', '#88898a', '#CCCCCC'),
 'Boston College': ('#98002e', '#bc9b6a', '#CCCCCC'),
 'Boston University': ('#cc0000', '#a3011b', '#CCCCCC'),
 'Bowling Green': ('#fe5000', '#4f2c1d', '#CCCCCC'),
 'Brown': ('#4e3629', '#7c2529', '#CCCCCC'),
 'Canisius': ('#0c2340', '#ffba00', '#CCCCCC'),
 'Clarkson': ('#0d433b', '#fac922', '#CCCCCC'),
 'Colgate': ('#821019', '#e10028', '#CCCCCC'),
 'Colorado College': ('#000000', '#f2af36', '#CCCCCC'),
 'Connecticut': ('#000e2f', '#ffffff', '#CCCCCC'),
 'Cornell': ('#b31b1b', '#5e3920', '#CCCCCC'),
 'Dartmouth': ('#046a38', '#000000', '#CCCCCC'),
 'Denver': ('#8b233b', '#8b6f4b', '#CCCCCC'),
 'Ferris State': ('#ba0c2f', '#fcc917', '#CCCCCC'),
 'Harvard': ('#a41034', '#000000', '#CCCCCC'),
 'Holy Cross': ('#602d89', '#ffffff', '#CCCCCC'),
 'Lake Superior': ('#003f87', '#ffc61e', '#CCCCCC'),
 'Lindenwood': ('#B5A36A', '#101820\xa0', '#CCCCCC'),
 'Long Island': ('#69b3e7', '#ffc72c', '#CCCCCC'),
 'Maine': ('#003263', '#B0D7FF', '#AB0634'),
 'Mass Lowell': ('#003da5', '#c8102e', '#CCCCCC'),
 'Mass. Lowell': ('#003da5', '#c8102e', '#CCCCCC'),
 'Massachusetts': ('#971b2f', '#FFFFFF', '#CCCCCC'),
 'Mercyhurst': ('#07594d', '#182752', '#CCCCCC'),
 'Merrimack': ('#003768', '#fdb813', '#CCCCCC'),
 'Miami': ('#b61e2e', '#000000', '#CCCCCC'),
 'Michigan': ('#0027ac', '#ffcb05', '#CCCCCC'),
 'Michigan State': ('#18453b', '#ffffff', '#CCCCCC'),
 'Michigan Tech': ('#000000', '#ffcd00', '#CCCCCC'),
 'Minnesota': ('#7a0019', '#ffcc33', '#CCCCCC'),
 'Minnesota Duluth': ('#8e0a26', '#fab937', '#CCCCCC'),
 'Minnesota State': ('#480059', '#f7e400', '#CCCCCC'),
 'New Hampshire': ('#4.10E+43', '#bbbcbc', '#CCCCCC'),
 'Niagara': ('#582c83', '#ffffff', '#CCCCCC'),
 'North Dakota': ('#009a44', '#aaaead', '#CCCCCC'),
 'Northeastern': ('#000000', '#e50000', '#CCCCCC'),
 'Northern Michigan': ('#095339', '#ffc425', '#CCCCCC'),
 'Notre Dame': ('#0c2340', '#c99700', '#CCCCCC'),
 'Ohio State': ('#bb0000', '#666666', '#CCCCCC'),
 'Omaha': ('#000000', '#d71920', '#636568'),
 'Penn State': ('#001E44', '#ffffff', '#CCCCCC'),
 'Princeton': ('#ff671f', '#000000', '#CCCCCC'),
 'Providence': ('#8a8d8f', '#000000', '#ffffff'),
 'Quinnipiac': ('#0c2340', '#ffb81c', '#CCCCCC'),
 'Rensselaer': ('#232020', '#ffffff', '#CCCCCC'),
 'RIT': ('#f76902', '#ffffff', '#CCCCCC'),
 'Robert Morris': ('#14234b', '#a6192e', '#CCCCCC'),
 'Sacred Heart': ('#ce1141', '#b1b3b6', '#CCCCCC'),
 'St Cloud State': ('#a10209', '#000000', '#CCCCCC'),
 'St Lawrence': ('#654134', '#e7d1a0', '#CCCCCC'),
 'St Thomas': ('#512773', '#98999b', '#CCCCCC'),
 'St. Cloud State': ('#a10209', '#000000', '#CCCCCC'),
 'St. Lawrence': ('#654134', '#e7d1a0', '#CCCCCC'),
 'St. Thomas': ('#512773', '#98999b', '#CCCCCC'),
 'Stonehill': ('#2F2975', '#ffffff', '#CCCCCC'),
 'Union': ('#9a0000', '#ffffff', '#CCCCCC'),
 'Vermont': ('#154734', '#8b5b29', '#CCCCCC'),
 'Western Michigan': ('#6c4023', '#b5a167', '#CCCCCC'),
 'Wisconsin': ('#c5050c', '#ffffff', '#CCCCCC'),
 'Yale': ('#00356b', '#ffffff', '#CCCCCC')}

In [69]:
## File Paths
folder_prefix = ''

data_folder = os.path.join(folder_prefix, '..', 'data/') # Data Folder Path
temp_folder = os.path.join(folder_prefix,'..', 'TEMP/',) # Temp Folder Path
image_folder = os.path.join(folder_prefix, '..', 'images/') # Image Folder Path
logo_folder = os.path.join(folder_prefix, image_folder, 'logos/') # Logo Folder Path
conference_logo_folder = os.path.join(folder_prefix, logo_folder, 'conference') # Conference Logo Folder Path
export_folder = os.path.join(folder_prefix, image_folder, 'export/') # Export Folder Path
background_folder = os.path.join(folder_prefix, image_folder, 'background/') # Background Folder Path

# Other paths
school_info_path = os.path.join(data_folder, 'school_info', 'arena_school_info.csv') # School Info Path

In [70]:
## Load the database
conn = sqlite3.connect(recent_clean_db, isolation_level=None)

## Extract player_stats and convert TOI into seconds for easier calculations
player_stats = pd.read_sql_query("SELECT * FROM player_stats", conn)

### TOI to seconds - From MM:SS string to seconds integer
def convert_toi_to_seconds(toi_str):
    if pd.isna(toi_str):
        return None
    try:
        minutes, seconds = map(int, toi_str.split(':'))
        return minutes * 60 + seconds
    except ValueError:
        return None

player_stats['TOI'] = player_stats['TOI'].apply(convert_toi_to_seconds)

# Check Data
# print(player_stats.head())


In [71]:
#### Create a date column for each row based on the Game_ID to track change in ice time over the season
def get_date_from_game_id(game_id):
    if pd.isna(game_id):
        return None
    try:
        return str(game_id)[:10]

        # Convert to datetime object
        date_obj = datetime.strptime(game_id, '%Y-%m-%d')
        # Convert to desired format
        

    except ValueError:
        return None

# Create a new column 'Date' in the DataFrame
player_stats['Date'] = player_stats['Game_ID'].apply(get_date_from_game_id)

# Check Data
# print(player_stats.head())

## Get Line assignments from DB

In [72]:
### Call the line_chart db table to the a dataframe
line_chart = pd.read_sql_query("SELECT * FROM line_chart", conn)

# Group into F / D / G lines based on Position column into new pos_1 / pos_2 columns
# # pos_1 = F / D / G 
# Center, Left Wing, Right Wing = F
# left D, Right D = D
# Goalie = G

# pos_2 = C / L / R / D / G


## Assign positions in the new columns
def assign_positions(row):
    if row['Position'] in ['Center', 'Left Wing', 'Right Wing']:
        return pd.Series(['F', row['Position'][0]])
    elif row['Position'] in ['Left D', 'Right D']:
        return pd.Series(['D', 'D'])
    elif row['Position'] == 'Goalie':
        return pd.Series(['G', 'G'])
    else:
        return pd.Series([None, None])
line_chart[['pos_1', 'pos_2']] = line_chart.apply(assign_positions, axis=1)

# Check Data
# print(line_chart.head(20))




#### Merge Position Data in with the player_stats dataframe based on Team, Player, and Game_ID
merged_df = pd.merge(player_stats, line_chart[['Game_ID', 'Team', 'Player', 'Line', 'pos_1', 'pos_2']], on=['Game_ID', 'Team', 'Player'], how='left')

# Relabel the rows of extra skaters - they have NaN in Line pos1 / pos_2 columns - Replace with 'E'
# merged_df.loc[merged_df['pos_1'].isna(), 'pos_1'] = 'E'
# merged_df.loc[merged_df['pos_2'].isna(), 'pos_2'] = 'E'
merged_df['pos_1'].fillna('E', inplace=True)
merged_df['pos_2'].fillna('E', inplace=True)
merged_df['Line'].fillna('E', inplace=True)

# Remove any rows with TOI = 0
merged_df = merged_df[merged_df['TOI'] != 0]



C:\Users\jbanc\AppData\Local\Temp\ipykernel_21864\2609500140.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['pos_1'].fillna('E', inplace=True)
C:\Users\jbanc\AppData\Local\Temp\ipykernel_21864\2609500140.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

## Get player stats ytd table from DB Calculate Player stats normalized for per 60 minutes

In [73]:
player_ytd_stats = pd.read_sql_query("SELECT * FROM player_stats_ytd", conn)

# Check Data
print(player_ytd_stats.head())

# Calulate Goals per 60 mins using the TOI_second value, Assists per 60, Points per 60
player_ytd_stats['Goals_per_60'] = (player_ytd_stats['G'] / player_ytd_stats['TOI_sec']) * 3600
player_ytd_stats['Assists_per_60'] = (player_ytd_stats ['A'] / player_ytd_stats['TOI_sec']) * 3600
player_ytd_stats['Points_per_60'] = (player_ytd_stats['Pts'] / player_ytd_stats['TOI_sec']) * 3600

# Calculate Avg TOI per Game and convert to MM:SS format
player_ytd_stats['Avg_TOI_per_Game'] = player_ytd_stats['TOI_sec'] / player_ytd_stats['Games_Played']
player_ytd_stats['Avg_TOI_per_Game'] = pd.to_datetime(player_ytd_stats['Avg_TOI_per_Game'], unit='s').dt.strftime('%M:%S')
# Rename Column To 'TOI / G'
player_ytd_stats.rename(columns={'Avg_TOI_per_Game': 'Mins/G'}, inplace=True)
# Conver TOI_sec to "TOI" MMM:SS format for display
player_ytd_stats['Total TOI'] = pd.to_datetime(player_ytd_stats['TOI_sec'], unit='s').dt.strftime('%H:%M:%S')

# Rename Games Played to GP for spacer in header
player_ytd_stats['GP'] = player_ytd_stats['Games_Played']

# Check Data
# print(player_ytd_stats.head())

     Clean_Player              Team  G  A  Pts  PlusMinus  Shots  TOI_sec  \
0     Aaron Pionk  Minnesota-Duluth  0  0    0         -5     17  13454.0   
1  Aaron Schwartz        Quinnipiac  1  8    9          2     22   8718.0   
2   Aaron Trotter        St. Thomas  0  0    0          0      0      0.0   
3     Abram Wiebe      North Dakota  2  6    8          1     19  10899.0   
4     Adam Barone     Lake Superior  0  8    8          1     15  11833.0   

   PIM  FOW  FOL  Games_Played  FO%       TOI  
0    0  0.0  0.0            12  NaN  03:44:14  
1    4  0.0  0.0            10  NaN  02:25:18  
2    0  0.0  0.0             3  NaN  00:00:00  
3    2  0.0  0.0            10  NaN  03:01:39  
4    8  0.0  0.0            10  NaN  03:17:13  


In [74]:
# player_ytd_stats

In [75]:
## Sort and display the top 20 players by Points per 60 mins
# First Filter for players with at least 60 minutes played
filtered_players = player_ytd_stats[player_ytd_stats['TOI_sec'] >= 3600]

top_20_points_per_60 = filtered_players.sort_values(by='Points_per_60', ascending=False).head(20)
# print(top_20_points_per_60)

In [76]:
# Calc and display the top 20 players by Goals per 60 mins
top_20_goals_per_60 = filtered_players.sort_values(by='Goals_per_60', ascending=False).head(20)
print(top_20_goals_per_60)

              Clean_Player               Team   G   A  Pts  PlusMinus  Shots  \
591        Hayden Stavroff          Dartmouth   6   2    8          3     14   
1530          Will Horcoff           Michigan  11   5   16          3     39   
93         Austin Burnevik    St. Cloud State  10   4   14          0     42   
845         Justin Poirier              Maine   9   5   14          3     43   
629            JJ Wiebusch         Penn State  11   7   18          6     40   
1320            Ryan Smith              Miami   6   2    8          5     26   
1069   Maximilion Helgeson              Miami   6   2    8          9     25   
1442        Tommi Männistö     Michigan State   5   3    8          7     18   
1068       Maxime Pellerin              Omaha   5   3    8          2     19   
327          Cole Eiserman  Boston University   6   2    8         -3     32   
313   Christian Fitzgerald          Wisconsin   8   3   11          3     35   
377            Cruz Lucius      Arizona 

## Viz code start

In [77]:
# team_df.head()

In [78]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from pathlib import Path

def lighten_color(color, amount=0.6):
    try:
        r, g, b = mcolors.to_rgb(color)
    except ValueError:
        return (0.9, 0.9, 0.9)
    return (
        (1 - amount) + amount * r,
        (1 - amount) + amount * g,
        (1 - amount) + amount * b,
    )


def create_player_leaderboard_figure(
    player_df: pd.DataFrame,
    stat_col: str = "Points_per_60",
    extra_cols=None,
    top_n: int = 15,
    min_toi_sec: int = 0,
    output_path: str | Path = "leaderboard_test.png",
    title: str | None = None,
    team_color_mapping: dict | None = None,
    column_order: list[str] | None = None,  # <--- NEW
):
    """
    Create a stylized leaderboard image of players, colored by team.
    """
    if team_color_mapping is None:
        team_color_mapping = {}

    if extra_cols is None:
        extra_cols = ["Team", "G", "A", "Pts", "Games_Played"]

    if stat_col not in player_df.columns:
        raise ValueError(f"stat_col '{stat_col}' not found in player_df.columns")

    df = player_df.copy()

    # Optional TOI filter
    if min_toi_sec:
        df = df[df["TOI_sec"] >= min_toi_sec]

    # Drop rows with no value for the chosen stat
    df = df.dropna(subset=[stat_col])

    # Sort & keep top N
    df = df.sort_values(stat_col, ascending=False).head(top_n).copy()

    if df.empty:
        raise ValueError(
            "No rows to display after filtering. "
            "Check min_toi_sec, stat_col, or your filters."
        )

    # Add Rank column at the front
    df.insert(0, "Rank", range(1, len(df) + 1))

    # ---------- column order control ----------
    if column_order is not None:
        # Use exactly the order the user specifies (dropping missing ones)
        cols = [c for c in column_order if c in df.columns]
        # Make sure stat_col is present
        if stat_col not in cols and stat_col in df.columns:
            cols.append(stat_col)
    else:
        # Original behavior
        cols = ["Rank", "Clean_Player", "Team"]
        cols += [
            c for c in extra_cols
            if c not in ("Rank", "Clean_Player", "Team") and c in df.columns
        ]
        if stat_col not in cols:
            cols.append(stat_col)
    # -----------------------------------------

    display_df = df[cols].copy()

    # Round floats
    for c in display_df.columns:
        if pd.api.types.is_float_dtype(display_df[c]):
            display_df[c] = display_df[c].round(2)

    # Nicer labels
    col_labels = [
        c.replace("_", " ").replace("per 60", "/60")
        for c in display_df.columns
    ]

    cell_text = display_df.values.tolist()
    n_rows, n_cols = display_df.shape

    # Figure size (height based on rows; width roughly on #cols)
    row_height = 0.4
    fig_height = 1.5 + n_rows * row_height
    fig_width = max(10, n_cols * 1.0)   # a bit wider base than before

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        loc="upper center",
        cellLoc="center",
    )

    # Font
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    # --- auto size columns based on content ---
    fig.canvas.draw()  # needed so text size is known
    table.auto_set_column_width(col=list(range(n_cols)))
    # only stretch vertically
    table.scale(1, 1.4)
    # ------------------------------------------

    # Header styling
    for col in range(n_cols):
        cell = table[0, col]
        cell.set_facecolor("#333333")
        cell.set_text_props(color="white", weight="bold")

    # Row styling: team primary = fill, secondary = text
    for row in range(1, n_rows + 1):
        team_name = display_df.iloc[row - 1]["Team"]
        colors = team_color_mapping.get(team_name, ("#CCCCCC", "#000000", "#CCCCCC"))

        primary = colors[0] if len(colors) > 0 and colors[0] else "#CCCCCC"
        secondary = colors[1] if len(colors) > 1 and colors[1] else "#000000"

        row_face = primary
        text_color = secondary

        for col in range(n_cols):
            cell = table[row, col]
            cell.set_facecolor(row_face)
            cell.set_text_props(color=text_color)
            cell.set_edgecolor("white")

    if title is None:
        title = f"Top {top_n} skaters by {stat_col}"

    ax.set_title(title, fontsize=18, pad=20, weight="bold")

    fig.tight_layout()
    output_path = Path(output_path)
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return output_path, display_df


In [79]:
column_order = [
    "Rank",
    "Clean_Player",
    "Team",
    "Goals_per_60",
    "Mins/G",
    "Total TOI",
    
    "GP",
    "G",
    "A",
    "Pts",
    
]

output_path, top_table = create_player_leaderboard_figure(
    player_df=player_ytd_stats,
    stat_col="Goals_per_60",
    extra_cols=["Mins/G", "Total TOI", "Games_Played", "G", "A", "Pts"],
    top_n=20,
    min_toi_sec=1800,
    output_path="leaders_goals_per60.png",
    title="Goals per 60 Leaders (Minimum 30 min TOI)",
    team_color_mapping=team_color_mapping,
    column_order=column_order,
)
print(f"Leaderboard image saved to: {output_path}")


Leaderboard image saved to: leaders_goals_per60.png


In [80]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from pathlib import Path

# Helper from earlier (in case it's not already in your notebook)
def lighten_color(color, amount=0.6):
    """
    Lighten a hex/RGB color by mixing it with white.
    amount: 0 (no change) to 1 (white).
    """
    try:
        r, g, b = mcolors.to_rgb(color)
    except ValueError:
        # Fallback if the color is missing/bad
        return (0.9, 0.9, 0.9)

    return (
        (1 - amount) + amount * r,
        (1 - amount) + amount * g,
        (1 - amount) + amount * b,
    )


def create_player_leaderboard_figure(
    player_df: pd.DataFrame,
    stat_col: str = "Points_per_60",
    extra_cols=None,
    top_n: int = 15,
    min_toi_sec: int = 0,
    output_path: str | Path = "leaderboard.png",
    title: str | None = None,
    team_color_mapping: dict | None = None,
):
    """
    Create a stylized leaderboard image of players, colored by team.

    Parameters
    ----------
    player_df : DataFrame
        Player stats with at least ['Clean_Player', 'Team', 'TOI_sec', stat_col].
    stat_col : str
        Column to sort by (e.g. 'Goals_per_60', 'Assists_per_60', 'Points_per_60').
    extra_cols : list[str] | None
        Additional columns from player_df to display besides Rank, Clean_Player, Team.
        Example: ['G', 'A', 'Pts', 'Games_Played', 'TOI'].
    top_n : int
        Number of players to show.
    min_toi_sec : int
        Minimum TOI_sec filter to include a player (0 = no filter).
    output_path : str or Path
        Path to save the PNG.
    title : str | None
        Figure title. If None, a default based on stat_col/top_n is used.
    team_color_mapping : dict
        Dict mapping team name -> (primary_hex, secondary_hex, optional_third).

    Returns
    -------
    output_path : Path
        Where the image was saved.
    display_df : DataFrame
        The final table of players (for reference/export).
    """
    if team_color_mapping is None:
        team_color_mapping = {}

    if extra_cols is None:
        extra_cols = ["Team", "G", "A", "Pts", "Games_Played"]

    if stat_col not in player_df.columns:
        raise ValueError(f"stat_col '{stat_col}' not found in player_df.columns")

    df = player_df.copy()

    # Optional TOI filter
    if min_toi_sec:
        df = df[df["TOI_sec"] >= min_toi_sec]

    # Drop rows with no value for the chosen stat
    df = df.dropna(subset=[stat_col])

    # Sort & keep top N
    df = df.sort_values(stat_col, ascending=False).head(top_n).copy()

    # If after filtering we have nothing, bail out nicely
    if df.empty:
        raise ValueError(
            "No rows to display after filtering. "
            "Check min_toi_sec, stat_col, or your filters."
        )

    # Add Rank column at the front
    df.insert(0, "Rank", range(1, len(df) + 1))

    # Build the display column list
    cols = ["Rank", "Clean_Player", "Team"]
    # Add user-specified extras, but only those that exist
    cols += [
        c for c in extra_cols
        if c not in ("Rank", "Clean_Player", "Team") and c in df.columns
    ]
    # Ensure the stat column is at the end
    if stat_col not in cols:
        cols.append(stat_col)

    display_df = df[cols].copy()

    # Round float columns to two decimals for nicer display
    for c in display_df.columns:
        if pd.api.types.is_float_dtype(display_df[c]):
            display_df[c] = display_df[c].round(2)

    # Column labels for the header row (prettify a bit)
    col_labels = [
        c.replace("_", " ").replace("per 60", "/60")
        for c in display_df.columns
    ]

    cell_text = display_df.values.tolist()
    n_rows, n_cols = display_df.shape

    # Figure sizing heuristics: tweak to taste
    row_height = 0.4
    col_width = 1.3
    fig_height = 1.5 + n_rows * row_height
    fig_width = max(8, n_cols * col_width)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        loc="upper center",
        cellLoc="center",
    )

    # Font & row height scaling
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)

    # Header styling
    for col in range(n_cols):
        cell = table[0, col]
        cell.set_facecolor("#333333")
        cell.set_text_props(color="white", weight="bold")

    # Row styling using each team’s colors from the team_color_mapping
    for row in range(1, n_rows + 1):
        team_name = display_df.iloc[row - 1]["Team"]

        # mapping: {'Air Force': ('#003087', '#8a8d8f', '#CCCCCC'), ...}
        colors = team_color_mapping.get(team_name, ("#CCCCCC", "#000000", "#CCCCCC"))

        primary = colors[0] if len(colors) > 0 and colors[0] else "#CCCCCC"
        secondary = colors[1] if len(colors) > 1 and colors[1] else "#000000"

        row_face = primary        # fill color for the row
        text_color = secondary    # text color for that row

        for col in range(n_cols):
            cell = table[row, col]
            cell.set_facecolor(row_face)
            cell.set_text_props(color=text_color)
            cell.set_edgecolor("white")


    # Title
    if title is None:
        title = f"Top {top_n} skaters by {stat_col}"

    ax.set_title(title, fontsize=16, pad=20, weight="bold")

    fig.tight_layout()
    output_path = Path(output_path)
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return output_path, display_df


In [ ]:
player_df = player_ytd_stats  # whatever you called it

output_path, top_table = create_player_leaderboard_figure(
    player_df=player_df,
    stat_col="Goals_per_60",
    extra_cols=["Mins/G", "Total TOI", "Games_Played", "G", "A", "Pts"],
    top_n=10,
    min_toi_sec=1500,  # 25 minutes
    output_path="leaders_goals_per60.png",
    title="Goals per 60 Leaders (Minimum 25 min TOI)",
    team_color_mapping=team_color_mapping,
)

print("Saved image to:", output_path)
top_table.head()


Saved image to: leaders_goals_per60.png


,Rank,Clean_Player,Team,Mins/G,Total TOI,Games_Played,G,A,Pts,Goals_per_60
591,1,Hayden Stavroff,Dartmouth,15:39,01:02:38,4,6,2,8,5.75
1318,2,Ryan Schelling,Dartmouth,09:25,00:37:43,4,3,0,3,4.77
1530,3,Will Horcoff,Michigan,14:54,02:58:57,12,11,5,16,3.69
93,4,Austin Burnevik,St. Cloud State,15:09,02:46:41,11,10,4,14,3.60
556,5,Gio DiGiulian,Cornell,13:50,00:55:23,4,3,0,3,3.25


In [ ]:
# ## Helper Functions for visualization

# def lighten_color(color, amount=0.6):
#     """
#     Lighten a hex/RGB color by mixing it with white.
#     amount: 0 (no change) to 1 (white).
#     """
#     try:
#         r, g, b = mcolors.to_rgb(color)
#     except ValueError:
#         # Fallback if the color is missing/bad
#         return (0.9, 0.9, 0.9)

#     return (
#         (1 - amount) + amount * r,
#         (1 - amount) + amount * g,
#         (1 - amount) + amount * b,
#     )


# ############
# ### HOULD NOT NEED IMPORTING TEAM COLORS - JUST FOR TESTING
# ############
# # # Normalize and Standardize hex values from team colors table
# # def normalize_hex_color(hex_color):
# # # Read hex1 column as string and add leading zeros if needed
# #     if pd.isna(hex_color) or not isinstance(hex_color, str):
# #         return None
# #     hex_color = hex_color.strip()
# #     if len(hex_color) == 6:
# #         return f"#{hex_color}"
# #     elif len(hex_color) == 3:
# #         return f"#{hex_color[0]*2}{hex_color[1]*2}{hex_color[2]*2}"
# #     else:
# #         return None

# player_df = player_ytd_stats
# team_df = pd.read_csv(school_info_path)
# team_df['hex1'] = team_df['hex1'].apply(normalize_hex_color)
# # apply the same normalization to hex2 if needed
# team_df['hex2'] = team_df['hex2'].apply(normalize_hex_color)


In [ ]:
# def create_player_leaderboard_figure(
#     player_df: pd.DataFrame,
#     team_df: pd.DataFrame,
#     stat_col: str = "Points_per_60",
#     extra_cols=None,
#     top_n: int = 15,
#     min_toi_sec: int = 0,
#     output_path: str | Path = "leaderboard.png",
#     title: str | None = None,
# ):
#     """
#     Create a stylized leaderboard image of players, colored by team.

#     Parameters
#     ----------
#     player_df : DataFrame
#         Player stats with at least ['Clean_Player', 'Team', 'TOI_sec', stat_col].
#     team_df : DataFrame
#         Team info with at least ['School', 'hex1'] (and optionally 'hex2', 'logo_abv').
#     stat_col : str
#         Column to sort by (e.g. 'Goals_per_60', 'Assists_per_60', 'Points_per_60').
#     extra_cols : list[str] | None
#         Additional columns from player_df to display besides Rank, Clean_Player, Team.
#         Example: ['G', 'A', 'Pts', 'Games_Played', 'TOI'].
#     top_n : int
#         Number of players to show.
#     min_toi_sec : int
#         Minimum TOI_sec filter to include a player (0 = no filter).
#     output_path : str or Path
#         Path to save the PNG.
#     title : str | None
#         Figure title. If None, a default based on stat_col/top_n is used.

#     Returns
#     -------
#     output_path : Path
#         Where the image was saved.
#     display_df : DataFrame
#         The final table of players (for reference/export).
#     """
#     if extra_cols is None:
#         extra_cols = ["Team", "G", "A", "Pts", "Games_Played"]

#     if stat_col not in player_df.columns:
#         raise ValueError(f"stat_col '{stat_col}' not found in player_df.columns")

#     df = player_df.copy()

#     # Optional TOI filter
#     if min_toi_sec:
#         df = df[df["TOI_sec"] >= min_toi_sec]

#     # Drop rows with no value for the chosen stat
#     df = df.dropna(subset=[stat_col])

#     # Deduplicate school info to avoid double-joins (e.g. American Int'l appears twice)
#     school_info = (
#         team_df[["School", "hex1", "hex2", "logo_abv"]]
#         .drop_duplicates(subset=["School"])
#     )

#     # Merge team colors onto player data
#     merged = df.merge(
#         school_info,
#         left_on="Team",
#         right_on="School",
#         how="left",
#     )

#     # Sort & keep top N
#     merged = merged.sort_values(stat_col, ascending=False).head(top_n)

#     # Add Rank column at the front
#     merged.insert(0, "Rank", range(1, len(merged) + 1))

#     # Build the display column list
#     cols = ["Rank", "Clean_Player", "Team"]
#     # Add user-specified extras, but only those that exist
#     cols += [c for c in extra_cols if c not in ("Rank", "Clean_Player", "Team") and c in merged.columns]
#     # Ensure the stat column is at the end
#     if stat_col not in cols:
#         cols.append(stat_col)

#     # If after filtering we have nothing, bail out nicely
#     if merged.empty:
#         raise ValueError(
#             "No rows to display after filtering. "
#             "Check min_toi_sec, stat_col, or your filters."
#         )

#     display_df = merged[cols].copy()

#     # Round float columns to two decimals for nicer display
#     for c in display_df.columns:
#         if pd.api.types.is_float_dtype(display_df[c]):
#             display_df[c] = display_df[c].round(2)

#     # Column labels for the header row (prettify a bit)
#     col_labels = [
#         c.replace("_", " ").replace("per 60", "/60")
#         for c in display_df.columns
#     ]

#     cell_text = display_df.values.tolist()
#     n_rows, n_cols = display_df.shape
    
#     # Figure sizing heuristics: tweak to taste
#     row_height = 0.4
#     col_width = 1.3
#     fig_height = 1.5 + n_rows * row_height
#     fig_width = max(8, n_cols * col_width)

#     fig, ax = plt.subplots(figsize=(fig_width, fig_height))
#     ax.axis("off")

#     table = ax.table(
#         cellText=cell_text,
#         colLabels=col_labels,
#         loc="upper center",
#         cellLoc="center",
#     )

#     # Font & row height scaling
#     table.auto_set_font_size(False)
#     table.set_fontsize(10)
#     table.scale(1, 1.5)

#     # Header styling
#     for col in range(n_cols):
#         cell = table[0, col]
#         cell.set_facecolor("#333333")
#         cell.set_text_props(color="white", weight="bold")

#     # Row striping using each team’s colors from the team_color_mapping
#     for row in range(1, n_rows + 1):
#         team_name = display_df.iloc[row - 1]["Team"]
#         team_colors = team_color_mapping.get(team_name, ("#CCCCCC", "#EEEEEE"))
#         base_color = team_colors[0] if team_colors[0] else "#CCCCCC"
#         alt_color = team_colors[1] if team_colors[1] else lighten_color(base_color, amount=0.7)

#         for col in range(n_cols):
#             cell = table[row, col]
#             # Alternate row colors
#             if row % 2 == 1:
#                 cell.set_facecolor(base_color)
#             else:
#                 cell.set_facecolor(alt_color)

#             # Text color adjustment for readability
#             if row % 2 == 1:
#                 cell.set_text_props(color="white")
#             else:
#                 cell.set_text_props(color="black")



#     # Title
#     if title is None:
#         title = f"Top {top_n} skaters by {stat_col}"

#     ax.set_title(title, fontsize=16, pad=20, weight="bold")

#     fig.tight_layout()
#     output_path = Path(output_path)
#     fig.savefig(output_path, dpi=200, bbox_inches="tight")
#     plt.close(fig)

#     return output_path, display_df


In [ ]:
team_df.head()

NameError: name 'team_df' is not defined

In [ ]:
# player_df = player_ytd_stats
# # team_df = pd.read_csv(school_info_path) # loaded and preprocessed above

# # Example: Top 20 by Goals_per_60, showing some extra context columns
# output_path, top_table = create_player_leaderboard_figure(
#     player_df=player_df,
#     team_df=team_df,
#     stat_col="Goals_per_60",
#     extra_cols=["Mins/G", "Total TOI", "Games_Played","G", "A", "Pts"],
#     top_n=10,
#     min_toi_sec=1500,  # e.g. require at least 5 minutes of TOI to cut out total noise
#     output_path="leaders_goals_per60.png",
#     title="Goals per 60 Leaders (Minimum 25 min TOI)",
# )

# print("Saved image to:", output_path)
# top_table.head()
